# 06 — Benchmark Strategies Validation
This notebook runs production benchmark strategies (`buy_and_hold`, `momentum`, `mean_reversion`, `random_policy`) through `src.workflows.benchmark_workflow` and validates metric outputs against recomputed ground truth from equity curves.

In [1]:
from __future__ import annotations

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from src.utils.config_loader import resolve_config, load_yaml
from src.utils.seed import set_global_seed
from src.workflows.data_workflow import run_data_workflow
from src.workflows.benchmark_workflow import run_benchmark_workflow
from src.evaluation.metrics import compute_metrics
from src.utils.artifact_manager import ArtifactManager

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
CONFIG = resolve_config(root=str(ROOT))
SEED = int(CONFIG.get('training', {}).get('random_seed', 42))
set_global_seed(SEED, deterministic_torch=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PAIRS = list(CONFIG['data']['pairs'])
BENCHMARKS = ['buy_and_hold', 'momentum', 'mean_reversion', 'random_policy']
OUTPUTS_ROOT = str(ROOT / 'outputs')
am = ArtifactManager(OUTPUTS_ROOT)
print(f'Pairs={PAIRS}')
print(f'Benchmarks={BENCHMARKS}')

2026-03-13 03:22:36 | src.utils.seed | INFO | Global seed set to 42 (deterministic_torch=True)


Pairs=['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD']
Benchmarks=['buy_and_hold', 'momentum', 'mean_reversion', 'random_policy']


## Build evaluation datasets and run all benchmarks

In [2]:
data_results = run_data_workflow(CONFIG, pairs=PAIRS, root=str(ROOT))
test_data = {pair: data_results[pair]['test'] for pair in PAIRS}

benchmark_results = {}
for bench in BENCHMARKS:
    bench_cfg = load_yaml(ROOT / 'configs' / 'benchmarks' / f'{bench}.yaml')
    benchmark_results[bench] = run_benchmark_workflow(
        config=CONFIG,
        benchmark_name=bench,
        benchmark_config=bench_cfg,
        pairs=PAIRS,
        data=test_data,
        outputs_root=OUTPUTS_ROOT,
        split_name='test',
    )

rows = []
for bench, pair_metrics in benchmark_results.items():
    for pair, metrics in pair_metrics.items():
        rows.append({'benchmark': bench, 'pair': pair, **metrics})
results_df = pd.DataFrame(rows)
display(results_df[['benchmark', 'pair', 'cumulative_return', 'sharpe_ratio', 'max_drawdown', 'turnover']].head(20))

,benchmark,pair,cumulative_return,sharpe_ratio,max_drawdown,turnover
0,buy_and_hold,EURUSD,-0.011035,-1.846832,0.013598,0.100419
1,buy_and_hold,GBPUSD,-0.008695,-1.786120,0.012741,0.100316
2,buy_and_hold,USDJPY,-0.005532,-1.743205,0.007151,0.100277
3,buy_and_hold,AUDUSD,-0.007628,-1.429869,0.014343,0.100124
4,momentum,EURUSD,-0.020813,-3.573519,0.025726,9.398086
5,momentum,GBPUSD,-0.006806,-1.419168,0.009362,8.896391
6,momentum,USDJPY,0.002737,0.921153,0.004801,5.502138
7,momentum,AUDUSD,-0.011772,-2.257505,0.017282,9.402251
8,mean_reversion,EURUSD,0.002257,0.559289,0.008381,7.999160
9,mean_reversion,GBPUSD,-0.005801,-1.667665,0.008051,6.600020


## Validate metric correctness against recomputed ground truth
For each benchmark/pair run, we recompute metrics directly from saved equity curves and assert equality with persisted metric snapshots (within numerical tolerance).

In [ ]:
validation_rows = []
for bench in BENCHMARKS:
    for pair in PAIRS:
        run_dir = am.benchmark_result_dir(bench, pair)
        eq_path = run_dir / 'metrics' / 'test' / 'equity_curve.csv'
        if not eq_path.exists():
            continue
        eq_df = pd.read_csv(eq_path)
        equity = eq_df['equity_value'].to_numpy(dtype=float)

        recomputed = compute_metrics(
            equity_curve=equity,
            trade_log=None,
            periods_per_year=CONFIG['evaluation']['periods_per_year'],
            risk_free_rate=CONFIG['evaluation']['risk_free_rate'],
        )

        for metric_name in ['cumulative_return', 'sharpe_ratio', 'max_drawdown', 'turnover']:
            metric_file = run_dir / 'metrics' / 'test' / f'{metric_name}.csv'
            if not metric_file.exists():
                continue
            persisted = float(pd.read_csv(metric_file)['value'].iloc[0])
            delta = abs(recomputed[metric_name] - persisted)
            validation_rows.append({
                'benchmark': bench,
                'pair': pair,
                'metric': metric_name,
                'recomputed': recomputed[metric_name],
                'persisted': persisted,
                'abs_delta': delta,
            })
            assert delta < 1e-8, f'Metric mismatch for {bench}/{pair}/{metric_name}: {delta}'

validation_df = pd.DataFrame(validation_rows)
display(validation_df.head(20))
print('✅ Metric snapshot validation passed.')

## Comparative strategy plots and summary tables

In [ ]:
summary_dir = Path(OUTPUTS_ROOT) / 'results' / 'benchmarks' / 'summary'
summary_dir.mkdir(parents=True, exist_ok=True)

pair_focus = PAIRS[0]
focus_df = results_df[results_df['pair'] == pair_focus].copy()
focus_df = focus_df.sort_values('cumulative_return', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(focus_df['benchmark'], focus_df['cumulative_return'])
axes[0].set_title(f'Cumulative return ({pair_focus})')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(focus_df['benchmark'], focus_df['sharpe_ratio'])
axes[1].set_title(f'Sharpe ratio ({pair_focus})')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plot_path = summary_dir / f'benchmark_comparison_{pair_focus}.png'
plt.savefig(plot_path, dpi=140)
plt.show()

table_path = summary_dir / 'benchmark_metrics_all_pairs.csv'
results_df.to_csv(table_path, index=False)
print(f'✅ Saved benchmark summary table -> {table_path}')
print(f'✅ Saved benchmark comparison plot -> {plot_path}')